# 🌱 FarmTech Vision — Fase 6

## Sistema de Visão Computacional com YOLO e Redes Neurais Convolucionais

Projeto desenvolvido para a **Fase 6 — O Começo da Rede Neural**, do curso de Inteligência Artificial da FIAP.

### 🎯 Objetivo

Desenvolver uma solução de visão computacional capaz de reconhecer e diferenciar dois objetos utilizando um dataset próprio, realizando etapas de treinamento, validação, teste e comparação entre diferentes abordagens de redes neurais.

### 📦 Classes utilizadas

- **Classe 0:** Coca-Cola
- **Classe 1:** Cereal Nescau

### 📊 Dataset

O dataset contém **80 imagens**, distribuídas em:

- **64 imagens para treinamento**
- **8 imagens para validação**
- **8 imagens para teste**

As imagens foram rotuladas no formato YOLO para permitir o treinamento de um modelo de detecção de objetos.

### 👥 Integrantes

- Willians — RM: 573314
- Mauricio — RM: 571019
- Fernando — RM: 569974
- Matheus  — RM: 572385
- Felipe   — RM: 564288

In [ ]:
# Conecta o Google Colab ao Google Drive
from google.colab import drive

drive.mount('/content/drive')

## 1. Preparação do ambiente e acesso ao dataset

O Google Drive foi conectado ao ambiente do Google Colab para permitir o acesso ao dataset utilizado no projeto.

A base está organizada no formato YOLO, com diretórios separados para imagens e rótulos de **treinamento, validação e teste**.

In [ ]:
# Importa a biblioteca para manipulação de caminhos e arquivos
import os

# Caminho definitivo do dataset no Google Drive
DATASET_PATH = "/content/drive/MyDrive/Fase6_FIAP/FIAP_Fase6_YOLO/dataset"

# Caminhos das imagens
train_path = os.path.join(DATASET_PATH, "images", "train")
val_path = os.path.join(DATASET_PATH, "images", "val")
test_path = os.path.join(DATASET_PATH, "images", "test")

# Conta apenas arquivos de imagem
def contar_imagens(pasta):
    extensoes = (".jpg", ".jpeg", ".png")
    return len([
        arquivo for arquivo in os.listdir(pasta)
        if arquivo.lower().endswith(extensoes)
    ])

print("Estrutura do dataset:")
print("Treinamento:", contar_imagens(train_path), "imagens")
print("Validação:", contar_imagens(val_path), "imagens")
print("Teste:", contar_imagens(test_path), "imagens")
print("Total:",
      contar_imagens(train_path) +
      contar_imagens(val_path) +
      contar_imagens(test_path),
      "imagens")

In [ ]:
# Caminho do arquivo de configuração do dataset
DATA_YAML = os.path.join(DATASET_PATH, "data.yaml")

# Exibe a configuração utilizada pelo YOLO
with open(DATA_YAML, "r", encoding="utf-8") as arquivo:
    print(arquivo.read())

## 2. Treinamento do modelo YOLO customizado

Nesta etapa será realizado o treinamento de um modelo YOLO utilizando o dataset próprio com as classes **Coca-Cola** e **Cereal**.

Serão realizados treinamentos com diferentes números de épocas para permitir a comparação das métricas e avaliar o impacto do tempo de treinamento no desempenho do modelo.

In [ ]:
# Instala a biblioteca Ultralytics, que contém o YOLO
!pip install -q ultralytics

In [ ]:
# Importa o YOLO e registra a versão da biblioteca utilizada
import ultralytics
from ultralytics import YOLO

print("Ultralytics:", ultralytics.__version__)
print("YOLO importado com sucesso!")

In [ ]:
# Simulação 1 - Treinamento YOLO com 30 épocas

modelo_30 = YOLO("yolo11n.pt")

resultado_30 = modelo_30.train(
    data=DATA_YAML,
    epochs=30,
    imgsz=640,
    batch=8,
    device=0,
    project="/content/drive/MyDrive/Fase6_FIAP/FIAP_Fase6_YOLO/resultados",
    name="yolo_30_epocas"
)

In [ ]:
# Simulação 2 - Treinamento YOLO com 60 épocas

modelo_60 = YOLO("yolo11n.pt")

resultado_60 = modelo_60.train(
    data=DATA_YAML,
    epochs=60,
    imgsz=640,
    batch=8,
    device=0,
    project="/content/drive/MyDrive/Fase6_FIAP/FIAP_Fase6_YOLO/resultados",
    name="yolo_60_epocas"
)

### 📊 Comparação dos treinamentos — 30 e 60 épocas

Foram realizados treinamentos com diferentes números de épocas para avaliar o impacto do tempo de treinamento no desempenho do modelo YOLO.

| Treinamento | Precision | Recall | mAP50 | mAP50-95 |
|---|---:|---:|---:|---:|
| 30 épocas | 0.956 | 0.936 | 0.936 | 0.736 |
| 60 épocas | 0.957 | 0.927 | 0.938 | 0.786 |

Os resultados mostram que o aumento de 30 para 60 épocas manteve o desempenho do modelo elevado. O mAP50 permaneceu praticamente estável, passando de 0.936 para 0.938, enquanto o mAP50-95 aumentou de 0.736 para 0.786.

Esse resultado indica uma melhora na capacidade do modelo de localizar os objetos com maior precisão considerando diferentes limiares de IoU. Dessa forma, o treinamento com 60 épocas apresentou melhor desempenho geral no mAP50-95.

In [ ]:
# Teste do melhor modelo treinado com 60 épocas
# Utiliza as 8 imagens reservadas para teste

from ultralytics import YOLO
import os

# Caminhos
BEST_MODEL = "/content/drive/MyDrive/Fase6_FIAP/FIAP_Fase6_YOLO/resultados/yolo_60_epocas/weights/best.pt"
TEST_IMAGES = "/content/drive/MyDrive/Fase6_FIAP/FIAP_Fase6_YOLO/dataset/images/test"
RESULTADOS_TESTE = "/content/drive/MyDrive/Fase6_FIAP/FIAP_Fase6_YOLO/resultados"

# Carrega o melhor modelo obtido no treinamento de 60 épocas
modelo_teste = YOLO(BEST_MODEL)

# Executa a detecção nas imagens de teste
resultados_teste = modelo_teste.predict(
    source=TEST_IMAGES,
    imgsz=640,
    conf=0.25,
    save=True,
    project=RESULTADOS_TESTE,
    name="teste_60_epocas"
)

print("Teste concluído com sucesso!")
print("Imagens processadas:", len(resultados_teste))
print("Resultados salvos em:")
print(os.path.join(RESULTADOS_TESTE, "teste_60_epocas"))

In [ ]:
# Exibe as imagens de teste com as detecções realizadas pelo modelo de 60 épocas

import os
import glob
import matplotlib.pyplot as plt
from PIL import Image

pasta_resultados = "/content/drive/MyDrive/Fase6_FIAP/FIAP_Fase6_YOLO/resultados/teste_60_epocas"

imagens_resultado = []

for extensao in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
    imagens_resultado.extend(glob.glob(os.path.join(pasta_resultados, extensao)))

imagens_resultado = sorted(imagens_resultado)

print("Imagens encontradas:", len(imagens_resultado))

plt.figure(figsize=(16, 12))

for i, caminho in enumerate(imagens_resultado):
    img = Image.open(caminho)

    plt.subplot(2, 4, i + 1)
    plt.imshow(img)
    plt.title(f"Teste {i + 1}")
    plt.axis("off")

plt.tight_layout()
plt.show()

### Análise qualitativa das detecções no conjunto de teste

Após o treinamento do modelo YOLO com **60 épocas**, foram realizadas inferências nas **8 imagens do conjunto de teste**, utilizando os pesos do melhor modelo obtido (`best.pt`).

A análise visual das imagens demonstra que o modelo foi capaz de detectar as duas classes propostas no projeto: **Coca_Cola** e **Cereal**. As detecções ocorreram em diferentes condições de posicionamento, distância e orientação dos objetos.

Também foi possível observar a identificação de **múltiplas instâncias de Coca-Cola em uma mesma imagem**, demonstrando a capacidade do modelo de realizar detecção de vários objetos simultaneamente.

Os resultados quantitativos do treinamento de 60 épocas foram:

- **Precision:** 0.957
- **Recall:** 0.927
- **mAP50:** 0.938
- **mAP50-95:** 0.786

De forma geral, os resultados quantitativos e a inspeção visual das imagens de teste indicam que o modelo apresentou bom desempenho na identificação e localização dos objetos das duas classes utilizadas no dataset.

In [ ]:
# Exibe os gráficos de desempenho do treinamento com 60 épocas

from PIL import Image
import matplotlib.pyplot as plt
import os

caminho_results = "/content/drive/MyDrive/Fase6_FIAP/FIAP_Fase6_YOLO/resultados/yolo_60_epocas/results.png"

if os.path.exists(caminho_results):
    img = Image.open(caminho_results)

    plt.figure(figsize=(16, 8))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Resultados do treinamento YOLO - 60 épocas")
    plt.show()
else:
    print("Arquivo results.png não encontrado.")

### Análise das curvas de treinamento

As curvas de treinamento do modelo YOLO com **60 épocas** demonstram uma evolução consistente ao longo do processo de aprendizagem. Observa-se uma tendência de redução das perdas (`box_loss`, `cls_loss` e `dfl_loss`) tanto no treinamento quanto na validação, indicando melhora progressiva na localização e classificação dos objetos.

As métricas de **Precision**, **Recall** e **mAP50** apresentaram crescimento principalmente nas primeiras épocas e posteriormente se estabilizaram em valores elevados. O **mAP50-95** também apresentou evolução ao longo do treinamento, embora com algumas oscilações.

Ao final das 60 épocas, o modelo alcançou **Precision de 0.957**, **Recall de 0.927**, **mAP50 de 0.938** e **mAP50-95 de 0.786**.

Considerando que o dataset utilizado possui **80 imagens**, sendo 64 para treinamento, 8 para validação e 8 para teste, os resultados demonstram bom desempenho experimental. Entretanto, devido ao tamanho reduzido do conjunto de dados, as métricas devem ser interpretadas considerando essa limitação.

In [ ]:
# Exibe a matriz de confusão do modelo treinado com 60 épocas

from PIL import Image
import matplotlib.pyplot as plt
import os

caminho_matriz = "/content/drive/MyDrive/Fase6_FIAP/FIAP_Fase6_YOLO/resultados/yolo_60_epocas/confusion_matrix.png"

if os.path.exists(caminho_matriz):
    img = Image.open(caminho_matriz)

    plt.figure(figsize=(9, 8))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Matriz de Confusão - YOLO 60 épocas")
    plt.show()
else:
    print("Arquivo confusion_matrix.png não encontrado.")

In [ ]:
import os

pasta_resultados = "/content/drive/MyDrive/Fase6_FIAP/FIAP_Fase6_YOLO/resultados/yolo_60_epocas"

for arquivo in sorted(os.listdir(pasta_resultados)):
    if "confusion" in arquivo.lower():
        print(arquivo)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import os

arquivo_matriz_norm = os.path.join(
    pasta_resultados,
    "confusion_matrix_normalized.png"
)

img = Image.open(arquivo_matriz_norm)

plt.figure(figsize=(10, 8))
plt.imshow(img)
plt.axis("off")
plt.title("Matriz de Confusão Normalizada - YOLO 60 épocas")
plt.show()

Análise da matriz de confusão

A matriz de confusão normalizada permite analisar o comportamento do modelo na identificação das classes Coca_Cola e Cereal no conjunto de validação. Observa-se melhor desempenho para a classe Cereal, enquanto a classe Coca_Cola apresentou maior ocorrência de confusão entre as detecções.
A presença da categoria background está relacionada às detecções que não encontraram correspondência adequada com os objetos anotados, aspecto próprio da avaliação de modelos de detecção de objetos.
Esses resultados devem ser analisados em conjunto com as demais métricas do modelo. No treinamento de 60 épocas, foram obtidos Precision de 0,957, Recall de 0,927, mAP50 de 0,938 e mAP50-95 de 0,786, indicando bom desempenho geral do modelo.
Como o dataset utilizado possui apenas 80 imagens, divididas em 64 para treinamento, 8 para validação e 8 para teste, os resultados devem ser interpretados como um experimento acadêmico com conjunto de dados reduzido.

In [ ]:
from pathlib import Path

# Localizar os modelos treinados salvos no Google Drive
modelos = list(Path('/content/drive/MyDrive').rglob('best.pt'))

print(f'Modelos encontrados: {len(modelos)}')
for modelo in modelos:
    print(modelo)

In [ ]:
from pathlib import Path

# Localizar as imagens reservadas para teste
base = Path("/content/drive/MyDrive/Fase6_FIAP/FIAP_Fase6_YOLO")

extensoes = {".jpg", ".jpeg", ".png", ".webp"}

for pasta in base.rglob("*"):
    if pasta.is_dir() and pasta.name.lower() == "test":
        imagens = [
            arquivo for arquivo in pasta.iterdir()
            if arquivo.is_file()
            and arquivo.suffix.lower() in extensoes
        ]
        print(f"Pasta: {pasta}")
        print(f"Imagens encontradas: {len(imagens)}")
        print("-" * 40)

In [ ]:
from ultralytics import YOLO
from pathlib import Path

# Caminhos do projeto
base = Path("/content/drive/MyDrive/Fase6_FIAP/FIAP_Fase6_YOLO")

# Modelo treinado durante 60 épocas
modelo_path = base / "resultados/yolo_60_epocas/weights/best.pt"

# Imagens reservadas para teste
imagens_teste = base / "dataset/images/test"

# Pasta para salvar os resultados
saida = base / "resultados/testes_60_epocas"

# Carregar o modelo
modelo = YOLO(str(modelo_path))

# Realizar detecções nas 8 imagens
resultados = modelo.predict(
    source=str(imagens_teste),
    conf=0.25,
    imgsz=640,
    save=True,
    project=str(saida.parent),
    name=saida.name,
    exist_ok=True
)

print("Teste concluído!")
print("Imagens processadas:", len(resultados))
print("Resultados salvos em:", saida)


In [ ]:
%pip install -q ultralytics

## Resultados Visuais das Detecções — YOLO (60 épocas)

Nesta etapa, são apresentados os resultados visuais das inferências realizadas pelo modelo YOLO treinado durante 60 épocas, utilizando as 8 imagens reservadas para teste.

O objetivo é verificar visualmente a identificação e a localização dos objetos pertencentes às classes **Coca_Cola** e **Cereal**, por meio das caixas delimitadoras (*bounding boxes*) e dos índices de confiança atribuídos pelo modelo.

A análise visual complementa as métricas quantitativas de Precision, Recall e mAP, permitindo observar acertos, possíveis falhas de classificação e a qualidade da localização dos objetos nas imagens de teste.

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

# Pasta com os resultados das 8 imagens de teste
pasta_resultados = Path(
    "/content/drive/MyDrive/Fase6_FIAP/FIAP_Fase6_YOLO/resultados/testes_60_epocas"
)

# Localizar imagens com detecções
extensoes = {".jpg", ".jpeg", ".png", ".webp"}

imagens = sorted([
    arquivo for arquivo in pasta_resultados.iterdir()
    if arquivo.is_file()
    and arquivo.suffix.lower() in extensoes
]) if pasta_resultados.exists() else []

print(f"Imagens encontradas: {len(imagens)}")

if imagens:
    fig, axes = plt.subplots(2, 4, figsize=(18, 9))
    for ax in axes.flat:
        ax.axis("off")

    for ax, caminho in zip(axes.flat, imagens[:8]):
        with Image.open(caminho) as img:
            ax.imshow(img.copy())
        ax.set_title(caminho.name, fontsize=10)

    plt.suptitle(
        "Resultados das Detecções YOLO - 60 Épocas",
        fontsize=16
    )
    plt.tight_layout()
    plt.show()
else:
    print("Nenhuma imagem encontrada. Verifique a pasta.")

Análise visual das detecções — YOLO (60 épocas)

Após o treinamento do modelo YOLO durante 60 épocas, foram realizadas inferências em 8 imagens reservadas para teste, não utilizadas no treinamento.

Os resultados demonstram que o modelo conseguiu identificar objetos das classes Coca-Cola e Cereal em diferentes condições de posicionamento, distância e enquadramento.

Na inspeção visual, foram observadas detecções com caixas delimitadoras e valores de confiança, incluindo uma identificação da classe Cereal com confiança de 0,94.

Entretanto, algumas imagens não apresentaram detecções visíveis, indicando possíveis limitações do modelo em determinadas condições de captura.

Considerando que o conjunto de dados possui apenas 80 imagens, os resultados são promissores para um experimento acadêmico, mas ainda não permitem afirmar que o modelo apresenta desempenho consistente em situações reais.

Como melhorias futuras, recomenda-se ampliar o dataset, diversificar os ambientes e condições de iluminação e realizar novos testes com imagens inéditas.

Observações sobre falhas de detecção

Durante a avaliação visual, foram identificadas duas situações relevantes:

- Ausência de detecção: uma imagem contendo a embalagem Nescau não apresentou caixa delimitadora, indicando que o modelo não identificou o objeto acima do limiar de confiança estabelecido.
- Detecção redundante: uma imagem contendo uma garrafa de Coca-Cola apresentou duas caixas delimitadoras sobrepostas para o mesmo objeto, com diferentes níveis de confiança.

Esses resultados demonstram limitações pontuais do modelo, reforçando a importância de ampliar o conjunto de treinamento, diversificar as imagens e aprimorar a configuração das detecções em trabalhos futuros.

Conclusão — Entrega 1

O projeto FarmTech Vision demonstrou a aplicação prática de técnicas de visão computacional e aprendizado profundo na detecção de objetos utilizando o modelo YOLO.

Foi desenvolvido um conjunto de dados com 80 imagens, distribuídas entre as classes Coca-Cola e Cereal, organizadas em treinamento, validação e teste.

Após 60 épocas de treinamento, o modelo apresentou resultados satisfatórios para um experimento acadêmico, alcançando Precision de 0,957, Recall de 0,927, mAP50 de 0,938 e mAP50-95 de 0,786.

A avaliação visual das imagens de teste demonstrou a capacidade do modelo de reconhecer os produtos em diferentes condições, embora tenham sido observadas situações de ausência de detecção e caixas delimitadoras redundantes.

Conclui-se que os objetivos experimentais de treinamento e avaliação do detector foram alcançados, reconhecendo-se a necessidade de ampliar o dataset e realizar testes adicionais para melhorar sua capacidade de generalização.

## Entrega 2 — Comparação de Modelos de Visão Computacional

Nesta etapa do projeto FarmTech Vision, serão comparadas três abordagens de inteligência artificial aplicadas ao conjunto de imagens de Coca-Cola e Cereal Nescau:

1. YOLO customizada, desenvolvida na Entrega 1.
2. YOLO tradicional, utilizando um modelo pré-treinado.
3. Rede Neural Convolucional (CNN), construída e treinada do zero.

A comparação considerará a facilidade de utilização e integração, a precisão dos modelos, o tempo de treinamento e o tempo de inferência.

O objetivo é avaliar as vantagens, limitações e aplicações de cada abordagem, considerando as diferenças entre detecção de objetos e classificação de imagens.

In [ ]:
# Entrega 2 - YOLO tradicional (pré-treinada)

from ultralytics import YOLO
import time

# Carregar o modelo padrão pré-treinado
inicio = time.perf_counter()

modelo_padrao = YOLO("yolov8n.pt")

tempo_carregamento = time.perf_counter() - inicio

print("YOLO tradicional carregada com sucesso!")
print(f"Tempo de carregamento: {tempo_carregamento:.2f} segundos")
print(f"Quantidade de classes: {len(modelo_padrao.names)}")

In [ ]:
# Entrega 2 - Teste da YOLO tradicional

from pathlib import Path
import time

# Localizar as imagens de teste
pasta_teste = dataset / "images" / "test"

imagens_teste = sorted([
    arquivo for arquivo in pasta_teste.iterdir()
    if arquivo.suffix.lower() in [".jpg", ".jpeg", ".png"]
])

print(f"Imagens encontradas: {len(imagens_teste)}")

# Executar a YOLO tradicional nas imagens
inicio = time.perf_counter()

resultados_padrao = modelo_padrao.predict(
    source=[str(img) for img in imagens_teste],
    conf=0.25,
    verbose=False
)

tempo_total = time.perf_counter() - inicio

print("Teste concluído!")
print(f"Tempo total: {tempo_total:.2f} segundos")
print(f"Tempo médio: {tempo_total / len(imagens_teste):.3f} segundos por imagem")

for imagem, resultado in zip(imagens_teste, resultados_padrao):
    print(f"{imagem.name}: {len(resultado.boxes)} detecções")

In [ ]:
# Entrega 2 - Análise das classes detectadas pela YOLO padrão

for imagem, resultado in zip(imagens_teste, resultados_padrao):
    print(f"\nImagem: {imagem.name}")

    if len(resultado.boxes) == 0:
        print("Nenhum objeto detectado.")
        continue

    for caixa in resultado.boxes:
        classe = int(caixa.cls[0])
        confianca = float(caixa.conf[0])

        print(
            f"Objeto: {modelo_padrao.names[classe]} "
            f"| Confiança: {confianca:.2%}"
        )

# Tempo de inferência informado pelo próprio modelo
tempos = [
    r.speed["inference"]
    for r in resultados_padrao
]

print("\nTempo médio de inferência:",
      round(sum(tempos) / len(tempos), 2), "ms")

In [ ]:
# Entrega 2 - Visualização das detecções da YOLO tradicional

import matplotlib.pyplot as plt

# Exibir as 8 imagens com as detecções
fig, axes = plt.subplots(2, 4, figsize=(18, 9))

for ax, resultado, imagem in zip(
    axes.flatten(), resultados_padrao, imagens_teste
):
    imagem_anotada = resultado.plot()

    # Converter BGR para RGB
    imagem_rgb = imagem_anotada[:, :, ::-1]

    ax.imshow(imagem_rgb)
    ax.set_title(imagem.name[:22], fontsize=9)
    ax.axis("off")

plt.suptitle(
    "Entrega 2 - Detecções da YOLO Tradicional",
    fontsize=16
)

plt.tight_layout()
plt.show()

### Análise dos Resultados — YOLO Tradicional

O modelo YOLOv8n pré-treinado foi avaliado utilizando oito imagens do conjunto de teste do projeto FarmTech Vision.

O modelo realizou 18 detecções, com tempo médio de inferência reportado de 5,4 ms.

**Principais observações:**

- Identificação de garrafas Coca-Cola pela categoria genérica `bottle`.
- Classificações incorretas de embalagens de cereal como `book` e `teddy bear`.
- Ocorrência de falso positivo, classificando uma garrafa inclinada como `dog`.
- Ausência de detecção dos produtos em determinadas imagens.
- Identificação de elementos do cenário, como `bed` e `couch`.

**Conclusão parcial:**

O modelo pré-treinado demonstrou capacidade de reconhecer categorias genéricas de objetos, mas apresentou limitações para identificar os produtos específicos do projeto.

Esses resultados reforçam a importância do treinamento personalizado para aplicações que exigem reconhecimento de classes específicas, como Coca-Cola e Cereal Nescau.

A comparação definitiva entre os modelos dependerá da avaliação das métricas e dos resultados obtidos nas demais abordagens.

In [ ]:
from pathlib import Path

# Localizar imagens para treinamento da CNN
base = Path("/content/drive/MyDrive/Fase6_FIAP/dataset_organizado")

for divisao in ["train", "val", "test"]:
    pasta = base / divisao

    print(f"\n{divisao.upper()}")

    if pasta.exists():
        for item in sorted(pasta.iterdir()):
            if item.is_dir():
                quantidade = len([
                    arq for arq in item.rglob("*")
                    if arq.suffix.lower() in [".jpg", ".jpeg", ".png"]
                ])
                print(f"{item.name}: {quantidade} imagens")
            else:
                print(f"Arquivo: {item.name}")
    else:
        print("Pasta não encontrada")

## Entrega 2 — Rede Neural Convolucional (CNN)

Nesta etapa, será desenvolvida uma Rede Neural Convolucional (CNN) para classificação de imagens dos produtos Coca-Cola e Cereal Nescau.

O conjunto de dados contém 80 imagens, distribuídas igualmente entre as duas classes.

A divisão utilizada será:
- Treinamento: 64 imagens (80%).
- Validação: 8 imagens (10%).
- Teste: 8 imagens (10%).

A CNN será construída e treinada do zero, sem utilização de pesos pré-treinados.

Serão avaliados o desempenho de classificação, o tempo de treinamento e o tempo de inferência, permitindo a comparação com os modelos YOLO.

In [ ]:
# Entrega 2 - Preparação dos dados para a CNN

import tensorflow as tf
from pathlib import Path

base_cnn = Path(
    "/content/drive/MyDrive/Fase6_FIAP/dataset_organizado"
)

IMG_SIZE = (128, 128)
BATCH_SIZE = 8
SEED = 42

treino = tf.keras.utils.image_dataset_from_directory(
    str(base_cnn / "train"),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=True,
    seed=SEED
)

validacao = tf.keras.utils.image_dataset_from_directory(
    str(base_cnn / "val"),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

teste = tf.keras.utils.image_dataset_from_directory(
    str(base_cnn / "test"),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

print("\nClasses identificadas:", treino.class_names)
print("TensorFlow:", tf.__version__)
print("Dados preparados com sucesso!")

In [ ]:
# Entrega 2 - Construção da CNN do zero

from tensorflow.keras import layers, models

tf.keras.utils.set_random_seed(42)

modelo_cnn = models.Sequential([
    layers.Input(shape=(128, 128, 3)),

    layers.Rescaling(1./255),

    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),

    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),

    layers.Dense(1, activation="sigmoid")
])

modelo_cnn.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

modelo_cnn.summary()

In [ ]:
# Entrega 2 - Treinamento da CNN

import time
from tensorflow.keras.callbacks import EarlyStopping

parada_antecipada = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

inicio_treinamento = time.perf_counter()

historico_cnn = modelo_cnn.fit(
    treino,
    validation_data=validacao,
    epochs=30,
    callbacks=[parada_antecipada],
    verbose=1
)

tempo_treinamento_cnn = (
    time.perf_counter() - inicio_treinamento
)

print("\nTreinamento concluído!")
print(
    f"Tempo total: {tempo_treinamento_cnn:.2f} segundos"
)
print(
    f"Épocas executadas: {len(historico_cnn.history['loss'])}"
)

In [ ]:
import matplotlib.pyplot as plt

historico = historico_cnn.history

fig, eixos = plt.subplots(1, 2, figsize=(13, 5))

eixos[0].plot(historico["accuracy"], label="Treinamento")
eixos[0].plot(historico["val_accuracy"], label="Validação")
eixos[0].set_title("CNN - Evolução da Acurácia")
eixos[0].set_xlabel("Épocas")
eixos[0].set_ylabel("Acurácia")
eixos[0].legend()
eixos[0].grid(True)

eixos[1].plot(historico["loss"], label="Treinamento")
eixos[1].plot(historico["val_loss"], label="Validação")
eixos[1].set_title("CNN - Evolução da Perda")
eixos[1].set_xlabel("Épocas")
eixos[1].set_ylabel("Loss")
eixos[1].legend()
eixos[1].grid(True)

plt.tight_layout()
plt.savefig("cnn_treinamento.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# Entrega 2 - Avaliação da CNN no conjunto de teste

perda_teste, acuracia_teste = modelo_cnn.evaluate(
    teste,
    verbose=1
)

print("\n=== RESULTADOS DA CNN ===")
print(f"Acurácia no teste: {acuracia_teste * 100:.2f}%")
print(f"Perda no teste: {perda_teste:.4f}")
print("Quantidade de imagens: 8")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Obter previsões e rótulos reais
y_real = []
y_previsto = []

for imagens, rotulos in teste:
    previsoes = modelo_cnn.predict(imagens, verbose=0)
    y_real.extend(rotulos.numpy().astype(int).ravel())
    y_previsto.extend((previsoes.ravel() >= 0.5).astype(int))

# Construir matriz de confusão
matriz = confusion_matrix(
    y_real, y_previsto, labels=[0, 1]
)

In [ ]:
# Exibir a matriz de confusão da CNN

disp = ConfusionMatrixDisplay(
    confusion_matrix=matriz,
    display_labels=["Coca-Cola", "Nescau"]
)

disp.plot(cmap="Blues", values_format="d")
plt.title("CNN - Matriz de Confusão")
plt.show()

## Análise dos Resultados — CNN

A rede neural convolucional (CNN) foi avaliada utilizando oito imagens do conjunto de teste, sendo quatro da classe Coca-Cola e quatro da classe Nescau.

O modelo alcançou acurácia de 100% no conjunto de teste, com perda de 0,1603.

A matriz de confusão apresentou oito classificações corretas e nenhuma classificação incorreta, demonstrando que a CNN conseguiu distinguir as duas classes nas imagens avaliadas.

Durante o treinamento, foram executadas 15 épocas, com oscilações nas métricas de validação, indicando possíveis limitações de generalização.

Embora os resultados no teste tenham sido positivos, o conjunto de apenas oito imagens é pequeno. Portanto, são necessários testes adicionais com imagens inéditas e condições variadas de iluminação, posicionamento e cenário para avaliar a robustez do modelo.

Esses resultados demonstram o potencial da CNN para classificação dos produtos Coca-Cola e Nescau, respeitando as limitações do conjunto de dados utilizado.

## Comparação Final dos Modelos — Entrega 2

Nesta etapa, são comparadas três abordagens de visão computacional utilizadas no projeto FarmTech Vision: YOLOv8 pré-treinado, YOLO personalizado e rede neural convolucional (CNN).

| Critério | YOLO tradicional | YOLO personalizado | CNN |
|---|---|---|---|
| Abordagem | Detecção genérica | Detecção personalizada | Classificação de imagens |
| Treinamento | Pré-treinado | 60 épocas | 15 épocas |
| Resultado | 18 detecções nas 8 imagens | Precision: 0,957; Recall: 0,927 | Acurácia no teste: 100% |
| Outras métricas | Inferência média: 5,4 ms | mAP50: 0,938; mAP50-95: 0,786 | Perda no teste: 0,1603 |
| Limitações | Confusão entre categorias genéricas | Falhas pontuais e detecções redundantes | Teste com apenas 8 imagens |

As métricas não são diretamente equivalentes, pois os modelos YOLO realizam detecção e localização de objetos, enquanto a CNN realiza classificação de imagens.

Análise Comparativa dos Modelos

A comparação entre as três abordagens evidencia diferenças importantes quanto à finalidade, ao desempenho e às limitações de cada modelo.

O YOLOv8 pré-treinado demonstrou capacidade de identificar categorias genéricas, mas apresentou dificuldades no reconhecimento específico dos produtos Coca-Cola e Cereal Nescau, incluindo classificações incorretas.

O YOLO personalizado, treinado durante 60 épocas, alcançou Precision de 0,957, Recall de 0,927, mAP50 de 0,938 e mAP50-95 de 0,786. Esses resultados indicam desempenho satisfatório na identificação e localização dos produtos, embora tenham sido observadas falhas pontuais nas detecções.

A CNN apresentou acurácia de 100% nas oito imagens de teste, classificando corretamente quatro imagens de Coca-Cola e quatro de Nescau. Entretanto, esse resultado deve ser interpretado com cautela devido ao tamanho reduzido do conjunto de teste e às oscilações observadas durante a validação.

É importante destacar que a CNN realiza classificação de imagens, enquanto os modelos YOLO realizam detecção e localização de objetos. Portanto, suas métricas não devem ser comparadas diretamente.

Conclusão — Entrega 2

O projeto FarmTech Vision permitiu implementar e avaliar diferentes técnicas de visão computacional, demonstrando a importância da escolha da arquitetura conforme o problema a ser resolvido.

Para aplicações que exigem a localização de produtos em imagens, o YOLO personalizado mostrou-se a alternativa mais adequada entre as abordagens avaliadas. Para tarefas de classificação de imagens contendo os produtos estudados, a CNN apresentou resultados positivos no conjunto de teste.

Apesar dos resultados promissores, o dataset de 80 imagens representa uma limitação experimental. Como trabalhos futuros, recomenda-se ampliar a base de dados, diversificar os cenários de captura e realizar avaliações com novas imagens.

Conclui-se que os experimentos atenderam aos objetivos de comparar as abordagens de detecção e classificação, evidenciando suas aplicações, vantagens e limitações.

Comparação dos Tempos de Treinamento — YOLO

Os treinamentos do modelo YOLO personalizado foram realizados com 30 e 60 épocas, permitindo avaliar a relação entre o tempo de processamento e o desempenho do modelo.

| Treinamento | Tempo registrado | mAP50 | mAP50-95 |
|---|---|---|---|
| 30 épocas | 129,65 segundos | 0,936 | 0,736 |
| 60 épocas | 221,06 segundos | 0,938 | 0,786 |

O treinamento de 60 épocas apresentou aumento de aproximadamente 70,5% no tempo registrado, acompanhado de uma melhoria no mAP50-95, de 0,736 para 0,786.

Os resultados indicam que o treinamento mais longo proporcionou melhor desempenho na localização dos objetos considerando diferentes limiares de IoU, embora tenha exigido maior tempo de processamento.

Os tempos foram obtidos a partir da coluna "time" dos arquivos "results.csv" gerados pelo Ultralytics YOLO.

## Tempo de Inferência — YOLO Personalizado

O modelo YOLO personalizado, treinado durante 60 épocas, foi avaliado utilizando as oito imagens reservadas para teste. O tempo médio de inferência registrado foi de 4,76 ms por imagem, no ambiente de execução do Google Colab.

Esse resultado indica processamento rápido nas condições avaliadas. Entretanto, o tempo pode variar conforme o hardware, a configuração do ambiente e as características das imagens. A medição refere-se à etapa de inferência, não ao tempo total de processamento.